In [13]:
from IPython.display import HTML, display

def show(df, height=350):
    html = f'<div style="height:{height}px; overflow:auto; border:1px solid #ccc;">{df.to_html()}</div>'
    display(HTML(html))

In [14]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from dotenv import load_dotenv
from supabase import create_client, Client
from datetime import date, datetime
from typing import Optional

load_dotenv('../.env')
print('SUPABASE_URL loaded:         ', bool(os.environ.get('SUPABASE_URL')))
print('SUPABASE_SERVICE_KEY loaded: ', bool(os.environ.get('SUPABASE_SERVICE_KEY')))

SUPABASE_URL loaded:          True
SUPABASE_SERVICE_KEY loaded:  True


1. Supabase Client

In [15]:
_client: Client | None = None

def _get_client() -> Client:
    global _client
    if _client is None:
        _client = create_client(
            os.environ['SUPABASE_URL'],
            os.environ['SUPABASE_SERVICE_KEY'],
        )
    return _client

client = _get_client()
print('Supabase client:', client)

Supabase client: <supabase._sync.client.Client object at 0x000001431957F410>


2. Date Helpers

`_parse_date` converts the raw date strings from bank PDFs (e.g. `'Oct 3'`, `'10/03'`) into a proper `date` object with the correct year attached.

In [16]:
_DATE_FORMATS = ['%b %d', '%B %d', '%m/%d', '%m/%d/%Y', '%m/%d/%y']

def _parse_date(date_str: str, year: int) -> Optional[date]:
    if not date_str or str(date_str).lower() in ('nan', 'none', ''):
        return None
    date_str = str(date_str).strip()
    for fmt in _DATE_FORMATS:
        try:
            parsed = datetime.strptime(date_str, fmt)
            return parsed.replace(year=year).date()
        except ValueError:
            continue
    return None

# Test cases
test_dates = [
    ('Oct 3',    2025),
    ('October 3', 2025),
    ('10/03',    2025),
    ('10/03/25', 2025),
    ('',         2025),   # empty -> None
    ('nan',      2025),   # NaN string -> None
    ('garbage',  2025),   # unparseable -> None
]
for raw, year in test_dates:
    print(f'{raw!r:15} year={year} -> {_parse_date(raw, year)}')

'Oct 3'         year=2025 -> 2025-10-03
'October 3'     year=2025 -> 2025-10-03
'10/03'         year=2025 -> 2025-10-03
'10/03/25'      year=2025 -> 2025-10-03
''              year=2025 -> None
'nan'           year=2025 -> None
'garbage'       year=2025 -> None


3. Amount / Type Helper

Bank CSVs use sign to indicate direction: negative = money coming in (credit/refund), positive = money going out (debit/purchase).

In [17]:
def _to_type(amount: float) -> str:
    return 'credit' if amount < 0 else 'debit'

# Test cases
for amount in [-7.89, -100.00, 0.00, 15.59, 1192.76]:
    print(f'{amount:>10.2f} -> {_to_type(amount)}')

     -7.89 -> credit
   -100.00 -> credit
      0.00 -> debit
     15.59 -> debit
   1192.76 -> debit


4. Account Helpers

`_get_or_create_account` looks up an account by `last_four` (or `account_name` if no last four). If it doesn't exist yet it inserts a new row and returns the new UUID. This means the same card is never duplicated across multiple statement uploads.

In [18]:
def _get_or_create_account(
    client: Client,
    user_id: str,
    account_name: str,
    account_type: str,
    last_four: Optional[str] = None,
) -> str:
    if last_four:
        res = (
            client.table('accounts')
            .select('bank_acc_id')
            .eq('user_id', user_id)
            .eq('last_four', last_four)
            .execute()
        )
    else:
        res = (
            client.table('accounts')
            .select('bank_acc_id')
            .eq('user_id', user_id)
            .eq('account_name', account_name)
            .execute()
        )

    if res.data:
        print(f'Account already exists: {res.data[0]["bank_acc_id"]}')
        return res.data[0]['bank_acc_id']

    res = client.table('accounts').insert({
        'user_id': user_id,
        'account_name': account_name,
        'account_type': account_type,
        'last_four': last_four,
    }).execute()
    print(f'Created new account: {res.data[0]["bank_acc_id"]}')
    return res.data[0]['bank_acc_id']

In [19]:
# Peek at what accounts already exist
res = client.table('accounts').select('*').execute()
pd.DataFrame(res.data)

,bank_acc_id,user_id,account_name,account_type,last_four,created_at
0,9656427f-60db-4bd9-a5f2-1e48dec343d7,184cc8ea-2430-43d6-8111-1f6297096658,Capital One VentureOne ****2952,credit,2952,2026-05-02T20:18:30.082265+00:00


5. Statement Helpers

`_statement_exists` checks by `file_hash` (SHA-256 of the PDF bytes) so the same file can never be uploaded twice. `_create_statement` inserts the statement record and returns its UUID.

In [20]:
def _statement_exists(client: Client, file_hash: str) -> Optional[str]:
    res = (
        client.table('statements')
        .select('statements_id')
        .eq('file_hash', file_hash)
        .execute()
    )
    return res.data[0]['statements_id'] if res.data else None

def _create_statement(
    client: Client,
    user_id: str,
    account_id: str,
    filename: str,
    file_hash: str,
    storage_path: str,
    period_start: Optional[date],
    period_end: Optional[date],
) -> str:
    res = client.table('statements').insert({
        'user_id': user_id,
        'account_id': account_id,
        'filename': filename,
        'file_hash': file_hash,
        'storage_path': storage_path,
        'period_start': period_start.isoformat() if period_start else None,
        'period_end': period_end.isoformat() if period_end else None,
    }).execute()
    return res.data[0]['statements_id']

# Peek at existing statements
res = client.table('statements').select('*').execute()
pd.DataFrame(res.data)

,statements_id,user_id,account_id,filename,file_hash,storage_path,period_start,period_end,uploaded_at
0,37c11f57-8e2c-4525-bd1c-0c46229dbdf3,184cc8ea-2430-43d6-8111-1f6297096658,9656427f-60db-4bd9-a5f2-1e48dec343d7,Capital_One_102025_2952.pdf,6e50adcf28f66bfaaee09f7403e57977c76b0b94f45dec...,,2025-09-20,2025-10-20,2026-05-02T20:18:30.291324+00:00


6. Merchant Category Cache

`get_cached_categories_bulk` does a single Supabase query for all descriptions at once and returns a `{description: category}` dict. `cache_categories_bulk` upserts new mappings in one call — if a description already exists it overwrites, otherwise it inserts.

In [21]:
def get_cached_categories_bulk(descriptions: list[str]) -> dict[str, str]:
    if not descriptions:
        return {}
    res = (
        client.table('merchant_categories')
        .select('description, category')
        .in_('description', descriptions)
        .execute()
    )
    return {row['description']: row['category'] for row in res.data}

def cache_categories_bulk(items: dict[str, str]) -> None:
    if not items:
        return
    client.table('merchant_categories').upsert([
        {'description': desc, 'category': cat}
        for desc, cat in items.items()
    ]).execute()

# Peek at the cache
res = client.table('merchant_categories').select('*').execute()
show(pd.DataFrame(res.data))

,description,category,created_at
0,aliexpressSan MateoCA -,Shopping,2026-05-02T20:18:29.509648+00:00
1,GETYOURGUIDE TICKETSLONDONGBR -,Entertainment,2026-05-02T20:18:29.509648+00:00
2,WWW.VOXI.CO.UKVODAFONE LTD,Bills & Utilities,2026-05-02T20:18:29.509648+00:00
3,SumUp fresh meetcanning townGBR,Food & Drink,2026-05-02T20:18:29.509648+00:00
4,TFL TRAVEL CHTFL.GOV.UK/CP,Travel,2026-05-02T20:18:29.509648+00:00
5,CGFLTCGLondonGBR,Food & Drink,2026-05-02T20:18:29.509648+00:00
6,Zettle_ DOUBLE SEVEN HLondonGBR,Food & Drink,2026-05-02T20:18:29.509648+00:00
7,LOON FUNGLONDON W1D,Entertainment,2026-05-02T20:18:29.509648+00:00
8,TESCO STORES,Groceries,2026-05-02T20:18:29.509648+00:00
9,CHEAPSIDE LEONLONDONGBR,Food & Drink,2026-05-02T20:18:29.509648+00:00


In [27]:
# Test bulk lookup
test_descs = ['MCDONALDS LONDON', 'NETFLIX', 'TESCO SUPERSTORE', 'NOT IN CACHE', 'ASDA STORES LONDON', 'TESCO STORES']
result = get_cached_categories_bulk(test_descs)
for desc in test_descs:
    hit = result.get(desc, '<miss>')
    print(f'{desc!r:35} -> {hit}')

'MCDONALDS LONDON'                  -> <miss>
'NETFLIX'                           -> <miss>
'TESCO SUPERSTORE'                  -> <miss>
'NOT IN CACHE'                      -> <miss>
'ASDA STORES LONDON'                -> <miss>
'TESCO STORES'                      -> Groceries


7. `upload_transactions` — Full Pipeline

Orchestrates the full upload in one call:
```
pdf_path
    -> parse_pdf()         extract transactions + metadata
    -> categorize_dataframe()  add 'category' column
    -> compute SHA-256 hash    for dedup
    -> get/create account
    -> dedup check by file_hash
    -> create statement record
    -> parse dates, build transaction rows
    -> bulk insert into transactions table
    -> return { statement_id, account_id, inserted, skipped }
```

In [23]:
import hashlib
from pipeline.pdf_parser import parse_pdf
from pipeline.categorizer import categorize_dataframe

def upload_transactions(
    pdf_path: str,
    user_id: str,
    df: Optional[pd.DataFrame] = None,
    skip_if_exists: bool = True,
    storage_path: str = '',
) -> dict:
    parsed = parse_pdf(pdf_path)

    if df is None:
        df = categorize_dataframe(parsed['transactions'])

    pdf_hash     = hashlib.sha256(open(pdf_path, 'rb').read()).hexdigest()
    pdf_filename = os.path.basename(pdf_path)
    account_name = f"{parsed['card_name']} ****{parsed['last_four']}"
    account_type = parsed['account_type'] or 'credit'
    last_four    = parsed['last_four']
    period_start = date.fromisoformat(parsed['period_start']) if parsed['period_start'] else None
    period_end   = date.fromisoformat(parsed['period_end'])   if parsed['period_end']   else None
    year         = (period_end or period_start or date.today()).year

    print(f'PDF hash:  {pdf_hash}')
    print(f'Account:   {account_name}  ({account_type})')
    print(f'Period:    {period_start} → {period_end}')

    account_id = _get_or_create_account(client, user_id, account_name, account_type, last_four)

    if skip_if_exists:
        existing_id = _statement_exists(client, pdf_hash)
        if existing_id:
            print(f'Already uploaded (statement {existing_id}), skipping.')
            return {'statement_id': existing_id, 'account_id': account_id, 'inserted': 0, 'skipped': True}

    statement_id = _create_statement(
        client, user_id, account_id, pdf_filename, pdf_hash, storage_path, period_start, period_end
    )

    rows = []
    for row in df.to_dict('records'):
        trans_date = _parse_date(row.get('trans_date'), year)
        if trans_date is None:
            continue
        amount_raw = float(row['amount1'])
        category   = row.get('category')
        rows.append({
            'statement_id': statement_id,
            'user_id':      user_id,
            'date':         trans_date.isoformat(),
            'description':  str(row['description']).strip(),
            'amount':       abs(amount_raw),
            'type':         _to_type(amount_raw),
            'category':     category if pd.notna(category) and category != '' else None,
        })

    if rows:
        client.table('transactions').insert(rows).execute()

    print(f'Inserted {len(rows)} transactions into statement {statement_id}')
    return {'statement_id': statement_id, 'account_id': account_id, 'inserted': len(rows), 'skipped': False}

In [24]:
USER_ID  = '184cc8ea-2430-43d6-8111-1f6297096658'
PDF_PATH = '../data/Capital_One_102025_2952.pdf'

result = upload_transactions(PDF_PATH, USER_ID)
print(result)

PDF hash:  6e50adcf28f66bfaaee09f7403e57977c76b0b94f45dec133c2dc677e941ea72
Account:   Capital One VentureOne ****2952  (credit)
Period:    2025-09-20 → 2025-10-20
Account already exists: 9656427f-60db-4bd9-a5f2-1e48dec343d7
Already uploaded (statement 37c11f57-8e2c-4525-bd1c-0c46229dbdf3), skipping.
{'statement_id': '37c11f57-8e2c-4525-bd1c-0c46229dbdf3', 'account_id': '9656427f-60db-4bd9-a5f2-1e48dec343d7', 'inserted': 0, 'skipped': True}
